# scPoli 二级注释流程 Python版

基于一级校正结果 `cell_type_level1_corrected` 做 subset/recluster，并用 marker 得到二级注释列 `cell_type_level2`。
直接用原来的scpoli来

该文件的输出都在：/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0514_rename_level2/output_allhuman/work_0514 

# import 

In [ ]:
import anndata
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
sc.settings.verbosity=2
sc.settings.seed=1234
np.random.seed(1234)

In [ ]:
outdir="../../7.5_correct/0511_rename_noIAISR/output_allhuman/work_0513_2/"
adata = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_corrected_all.h5ad"))
adata

AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [ ]:
adata.obs['cell_type_level1_corrected'].value_counts()

cell_type_level1_corrected
T cell                   355512
Macrophage               104714
B cell                    89606
Fibroblast                87730
Neutrophil                83219
Smooth muscle cell        72481
Endothelial cell          65117
Natural killer cell       53467
Dendritic cell            34542
Monocyte                  33314
Pericyte                  18985
Mast cell                  6959
Erythrocyte/Erythroid      6884
Basophil                   3169
Name: count, dtype: int64

In [ ]:
adata.obsm["X_scPoli"].shape

(1015699, 10)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_scPoli")
sc.tl.umap(adata)

computing neighbors
    finished (0:03:26)
computing UMAP
    finished (0:23:11)


In [ ]:
##合并后先画一张全体细胞的 UMAP，检查 cell_type_level1_corrected 是否正常
outdir="../0514_rename_level2/output_allhuman/work_0514/"
if "X_umap" in adata.obsm and "cell_type_level1_corrected" in adata.obs:
    sc.pl.umap(adata,color="cell_type_level1_corrected",legend_loc="on data",frameon=False,size=1,show=False)
    plt.savefig(os.path.join(outdir,"cell_type_level1_corrected.pdf"),bbox_inches="tight")
    plt.close()
adata

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'distances', 'connectivities'

# mac

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/mac/"

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
mac = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Macrophage"],
    outdir=outdir,
    prefix="mac_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Macrophage']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 104714 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 's

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/mac/umap_mac_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/mac/umap_mac_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/mac/umap_mac_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     3610
1     3556
2     3524
3     3149
4     3062
5     3028
6     2928
7     2778
8     2776
9     2775
10    2755
11    2729
12    2646
13    2644
14    2584
15    2563
16    2559
17    2547
18    2507
19    2497
20    2452
21    2448
22    2421
23    2402
24    2239
25    2160
26    2120
27    1953
28    1936
29    1880
30    1867
31    1743
32    1742
33    1724
34    1671
35    1639
36    1619
37    1541
38    1478
39    1467
40    1400
41    1380
42    1306
43    1254
44    1115
45    1112
46     850
47     521
48      36
49      21
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/mac/mac_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "mac_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 104714 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Inflammatory macrophage": ['S100A8', 'IL1B', 'S100A9','IRF7', 'IFITM3', 'ISG15', 'IFIT2'],
    "Foamy macrophage": ['TREM2', 'MARCO', 'FABP4','FABP5', 'CD36'],
    "Homeostatic/Resident macrophage": ['LYVE1', 'MRC1', 'F13A1', 'SEPP1', 'IGF1', 'GAS6', 'MERTK', 'STAB1', 'C1QA', 'C1QB', 'C1QC', 'FOLR2'],
    "Proliferative macrophage": ['MKI67', 'TOP2A', 'PCNA', 'BIRC5', 'MCM2','MCM4','MCM5', 'TK1'],
    "Matrix-remodeling/SMC-like macrophage": ['AEBP1', 'COL1A1', 'COL1A2', 'THBS2', 'MMP3', 'S100A16', 'TNC', 'PDGFRB'],
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_mono_neu_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_mono_neu_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Inflammatory macrophage ['S100A8', 'IL1B', 'S100A9', 'IRF7', 'IFITM3', 'ISG15', 'IFIT2']
Foamy macrophage ['TREM2', 'MARCO', 'FABP4', 'FABP5', 'CD36']
Homeostatic/Resident macrophage ['LYVE1', 'MRC1', 'F13A1', 'IGF1', 'GAS6', 'MERTK', 'STAB1', 'C1QA', 'C1QB', 'C1QC', 'FOLR2']
Proliferative macrophage ['MKI67', 'TOP2A', 'PCNA', 'BIRC5', 'MCM2', 'MCM4', 'MCM5', 'TK1']
Matrix-remodeling/SMC-like macrophage ['AEBP1', 'COL1A1', 'COL1A2', 'THBS2', 'MMP3', 'S100A16', 'TNC', 'PDGFRB']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mac_mono_neu_cluster_level2_marker_summary.csv"))


                                  major_marker  major_marker_frac  n_cells                    cluster_label_clean
cluster                                                                                                          
0              Homeostatic/Resident macrophage           0.916620     3610        Homeostatic/Resident macrophage
1                             Foamy macrophage           0.790495     3556                       Foamy macrophage
2              Homeostatic/Resident macrophage           0.683314     3524        Homeostatic/Resident macrophage
3                      Inflammatory macrophage           0.633217     3149                Inflammatory macrophage
4                      Inflammatory macrophage           0.752123     3062                Inflammatory macrophage
5                             Foamy macrophage           0.360634     3028                       Foamy macrophage
6              Homeostatic/Resident macrophage           0.952527     2928        Homeos

In [ ]:
corrected_annotation = {
    "5": "Homeostatic/Resident macrophage",
    "23": "Homeostatic/Resident macrophage",
    "25": "Foamy macrophage",
    "34": "Inflammatory macrophage",
    "35": "Other macrophage",
    "39": "Inflammatory macrophage",
    "41": "Inflammatory macrophage",
    "44": "Matrix-remodeling/SMC-like macrophage",
    "48": "Other macrophage",
    "49": "Other macrophage"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Homeostatic/Resident macrophage          69184
Inflammatory macrophage                  20256
Foamy macrophage                         12463
Other macrophage                          1696
Matrix-remodeling/SMC-like macrophage     1115
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "mac_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mac_level2_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_mac_level2_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_mac_level2_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mac_level2_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_mac_level2_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# smc

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/smc"

In [ ]:
# adata = sc.read_h5ad("./output_allhuman/work_0513_2/scPoli_concat_level2_marker_all.h5ad")
# adata

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
smc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Smooth muscle cell"],
    outdir=outdir,
    prefix="smc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Smooth muscle cell']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 72481 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weight

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/smc/umap_smc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/smc/umap_smc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/smc/umap_smc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     2873
1     2670
2     2465
3     2428
4     2341
5     2286
6     2275
7     2164
8     2051
9     2040
10    2025
11    2018
12    1993
13    1951
14    1936
15    1894
16    1889
17    1880
18    1810
19    1727
20    1722
21    1658
22    1638
23    1601
24    1594
25    1578
26    1576
27    1440
28    1419
29    1403
30    1383
31    1278
32    1270
33    1225
34    1170
35    1130
36     953
37     930
38     891
39     852
40     836
41     743
42     716
43     448
44     236
45      75
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/smc/smc_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "smc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 72481 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Smooth muscle cell": ['MYH11','MYL9','TPM2','CALD1','TAGLN','APOE','APOC1','AGT','NOTCH3','PDGFRB','MFAP4'],#'LUM','TNFRSF11B','ACTA2',
    "Fibromyocyte": ['FN1','LUM','TNFRSF11B','ACTA2','TCF21']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_smc_fibro_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_smc_fibro_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Smooth muscle cell ['MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4']
Fibromyocyte ['FN1', 'LUM', 'TNFRSF11B', 'ACTA2', 'TCF21']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "smc_fibro_cluster_level2_marker_summary.csv"))


               major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                                    
0        Smooth muscle cell           0.997564     2873  Smooth muscle cell
1        Smooth muscle cell           0.989888     2670  Smooth muscle cell
2              Fibromyocyte           0.649899     2465        Fibromyocyte
3        Smooth muscle cell           0.952224     2428  Smooth muscle cell
4              Fibromyocyte           0.796241     2341        Fibromyocyte
5        Smooth muscle cell           0.827647     2286  Smooth muscle cell
6        Smooth muscle cell           0.964396     2275  Smooth muscle cell
7        Smooth muscle cell           0.989372     2164  Smooth muscle cell
8        Smooth muscle cell           0.854705     2051  Smooth muscle cell
9        Smooth muscle cell           0.986275     2040  Smooth muscle cell
10       Smooth muscle cell           0.984691     2025  Smooth muscle cell
11       Smo

In [ ]:
corrected_annotation = {
    "28": "Fibromyocyte",
    "34": "Fibromyocyte"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

## 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Smooth muscle cell    57935
Fibromyocyte          14546
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "smc_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_smc_fib_cell_type_level2.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_smc_fib_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_smc_fib_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_smc_fib_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_smc_fib_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# T cell

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/tc"

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
tc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["T cell"],
    outdir=outdir,
    prefix="tc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['T cell']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 355512 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDbl

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/tc/umap_tc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/tc/umap_tc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/tc/umap_tc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     12423
1     11055
2     10538
3     10473
4     10437
5     10436
6     10114
7      9971
8      9956
9      9632
10     9620
11     9552
12     9485
13     9410
14     9131
15     9073
16     8813
17     8715
18     8688
19     8421
20     8359
21     8188
22     8131
23     8067
24     7873
25     7553
26     7544
27     7414
28     7112
29     6641
30     6456
31     6447
32     6322
33     6259
34     6245
35     6143
36     5885
37     5865
38     5838
39     5100
40     4567
41     4243
42     3863
43     3597
44     2665
45     1951
46      856
47      309
48       76
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/tc/tc_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "tc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 355512 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "CD8 T cell":['CCL4L2', 'CRTAM', 'GZMK', 'CCL4', 'CD8A', 'GZMH'], 
    "CD4 T cell":['IL7R', 'ANXA1', 'FTH1', 'CD4', 'BATF', 'TNFRSF4', 'TNFRSF18']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_tc_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_tc_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
CD8 T cell ['CCL4L2', 'CRTAM', 'GZMK', 'CCL4', 'CD8A', 'GZMH']
CD4 T cell ['IL7R', 'ANXA1', 'FTH1', 'CD4', 'BATF', 'TNFRSF4', 'TNFRSF18']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "tc_cluster_level2_marker_summary.csv"))


        major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                             
0         CD4 T cell           0.997666    12423          CD4 T cell
1         CD4 T cell           0.995568    11055          CD4 T cell
2         CD4 T cell           0.998482    10538          CD4 T cell
3         CD8 T cell           0.960088    10473          CD8 T cell
4         CD4 T cell           0.890582    10437          CD4 T cell
5         CD4 T cell           0.999713    10436          CD4 T cell
6         CD4 T cell           0.953728    10114          CD4 T cell
7         CD4 T cell           0.988467     9971          CD4 T cell
8         CD8 T cell           0.794797     9956          CD8 T cell
9         CD4 T cell           0.567068     9632          CD4 T cell
10        CD8 T cell           0.681497     9620          CD8 T cell
11        CD4 T cell           0.982726     9552          CD4 T cell
12        CD8 T cell           0.9

In [ ]:
corrected_annotation = {
    "9" : "CD8 T cell"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
CD4 T cell    228073
CD8 T cell    127439
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "tc_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_tc_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_tc_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_tc_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_tc_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_tc_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# ECs

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/ec"

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
ec = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Endothelial cell"],
    outdir=outdir,
    prefix="ec_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Endothelial cell']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 65117 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/ec/umap_ec_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/ec/umap_ec_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/ec/umap_ec_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     2818
1     2597
2     2185
3     2069
4     2034
5     1954
6     1912
7     1886
8     1870
9     1852
10    1846
11    1843
12    1829
13    1773
14    1759
15    1735
16    1692
17    1550
18    1527
19    1515
20    1479
21    1479
22    1423
23    1373
24    1343
25    1340
26    1333
27    1330
28    1305
29    1269
30    1242
31    1205
32    1196
33    1189
34    1129
35    1029
36     981
37     951
38     945
39     930
40     865
41     839
42     429
43     170
44      97
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/ec/ec_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "ec_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 65117 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Arterial endothelial cell": ['GJ4', 'GJ5', "GATA2", "MECOM", "GJA4", "GJA5"],
    "Lymphatic endothelial cell": ["LYVE1", "CCL21"], 
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_ec_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_ec_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Arterial endothelial cell ['GATA2', 'MECOM', 'GJA4', 'GJA5']
Lymphatic endothelial cell ['LYVE1', 'CCL21']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "ec_cluster_level2_marker_summary.csv"))


                       major_marker  major_marker_frac  n_cells         cluster_label_clean
cluster                                                                                    
0        Lymphatic endothelial cell           0.647268     2818  Lymphatic endothelial cell
1         Arterial endothelial cell           0.884097     2597   Arterial endothelial cell
2        Lymphatic endothelial cell           0.607780     2185  Lymphatic endothelial cell
3         Arterial endothelial cell           0.668439     2069   Arterial endothelial cell
4        Lymphatic endothelial cell           0.638643     2034  Lymphatic endothelial cell
5        Lymphatic endothelial cell           0.580860     1954  Lymphatic endothelial cell
6        Lymphatic endothelial cell           0.612448     1912  Lymphatic endothelial cell
7         Arterial endothelial cell           0.596501     1886   Arterial endothelial cell
8         Arterial endothelial cell           0.631016     1870   Arterial endot

In [ ]:
corrected_annotation = {
    "44": "Arterial endothelial cell"
   
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Lymphatic endothelial cell    43449
Arterial endothelial cell     21668
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "ec_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_ec_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_ec_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_ec_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_ec_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_ec_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# dc

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/dc"

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
dc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Dendritic cell"],
    outdir=outdir,
    prefix="dc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Dendritic cell']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 34542 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted',

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/dc/umap_dc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/dc/umap_dc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/dc/umap_dc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     1709
1     1520
2     1433
3     1378
4     1351
5     1302
6     1273
7     1235
8     1177
9     1171
10    1163
11    1155
12    1101
13    1093
14    1004
15     936
16     917
17     896
18     873
19     790
20     774
21     723
22     722
23     712
24     696
25     680
26     677
27     631
28     622
29     617
30     594
31     583
32     505
33     487
34     480
35     419
36     417
37     406
38     280
39      40
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/dc/dc_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "dc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 34542 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "cDC1": ["CLEC9A", "IRF8", "SNX3"],
    "cDC2": ["CD1C", "CLEC10A", "FCER1A"],
    "Plasmacytoid dendritic cell": ["IRF7", "TLR7", "TLR9", "NRP1", "CLEC4C", "SCAMP5", "GZMB", "TCF4"]
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_dc_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_dc_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
cDC1 ['CLEC9A', 'IRF8', 'SNX3']
cDC2 ['CD1C', 'CLEC10A', 'FCER1A']
Plasmacytoid dendritic cell ['IRF7', 'TLR7', 'TLR9', 'NRP1', 'CLEC4C', 'SCAMP5', 'GZMB', 'TCF4']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "dc_cluster_level2_marker_summary.csv"))


                        major_marker  major_marker_frac  n_cells          cluster_label_clean
cluster                                                                                      
0                               cDC1           0.521943     1709                         cDC1
1                               cDC2           0.550000     1520                         cDC2
2        Plasmacytoid dendritic cell           0.679693     1433  Plasmacytoid dendritic cell
3                               cDC1           0.523948     1378                         cDC1
4                               cDC1           0.466321     1351                         cDC1
5        Plasmacytoid dendritic cell           0.639017     1302  Plasmacytoid dendritic cell
6                               cDC1           0.531029     1273                         cDC1
7                               cDC2           0.757895     1235                         cDC2
8                               cDC2           0.659303     

In [ ]:
corrected_annotation = {
    "1": "cDC1",
    "19": "cDC1",
    "23": "cDC1",
    "37": "Plasmacytoid dendritic cell",
    "39": "cDC1"
    
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
cDC1                           20734
cDC2                            9944
Plasmacytoid dendritic cell     3864
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "dc_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_dc_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_nk_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_dc_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_dc_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_nk_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# B cells

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/bc"

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
bc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["B cell"],
    outdir=outdir,
    prefix="bc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['B cell']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 89606 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblF

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/bc/umap_bc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/bc/umap_bc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/bc/umap_bc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     3361
1     2876
2     2869
3     2755
4     2751
5     2743
6     2644
7     2548
8     2532
9     2532
10    2529
11    2515
12    2511
13    2445
14    2335
15    2331
16    2283
17    2164
18    2152
19    2127
20    2096
21    2065
22    2029
23    2005
24    1957
25    1905
26    1869
27    1853
28    1763
29    1762
30    1742
31    1681
32    1674
33    1622
34    1597
35    1542
36    1496
37    1482
38    1452
39    1413
40    1289
41    1066
42     708
43     305
44     230
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/bc/bc_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "bc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 89606 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
   "B cell": ['CD79A', 'CD79B', 'MS4A1', 'CD22', 'FCER2'],#IGKC
    "Plasma cell": ['IGKC', 'IGHM', 'IGHA1', 'IGLC2', 'IGLC3', 'JCHAIN']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_bc_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_bc_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
B cell ['CD79A', 'CD79B', 'MS4A1', 'CD22', 'FCER2']
Plasma cell ['IGKC', 'IGHM', 'IGHA1', 'IGLC2', 'IGLC3', 'JCHAIN']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "bc_cluster_level2_marker_summary.csv"))


        major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                             
0        Plasma cell           0.645344     3361         Plasma cell
1        Plasma cell           0.998261     2876         Plasma cell
2        Plasma cell           0.997560     2869         Plasma cell
3             B cell           0.876225     2755              B cell
4             B cell           0.928026     2751              B cell
5             B cell           0.815166     2743              B cell
6             B cell           0.919062     2644              B cell
7             B cell           0.565934     2548              B cell
8             B cell           0.833333     2532              B cell
9             B cell           0.894155     2532              B cell
10            B cell           0.686042     2529              B cell
11            B cell           0.854076     2515              B cell
12            B cell           0.5

In [ ]:
corrected_annotation = {
    "0": "B cell",
    "16": "B cell",
    "21": "B cell",
    "35": "B cell",
    "37": "B cell",
    "41": "B cell",
    "43": "B cell",
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
B cell         78088
Plasma cell    11518
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "bc_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_bc_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_bc_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_bc_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_bc_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_bc_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# mono

In [ ]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/work_0514/mono/"

In [ ]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [ ]:
mono = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Monocyte"],
    outdir=outdir,
    prefix="mono_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Monocyte']
Original object: AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Subset object: AnnData object with n_obs × n_vars = 33314 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDb

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/mono/umap_mono_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/mono/umap_mono_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/work_0514/mono/umap_mono_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     1185
1     1182
2     1169
3     1150
4     1143
5     1114
6     1113
7     1108
8     1070
9     1061
10    1054
11    1052
12    1037
13     992
14     989
15     986
16     968
17     960
18     927
19     884
20     879
21     874
22     770
23     769
24     746
25     741
26     739
27     738
28     723
29     686
30     676
31     673
32     669
33     639
34     581
35     573
36     352
37     236
38     106
Name: count, dtype: int64
Saved to: ./output_allhuman/work_0514/mono/mono_level2_scPoli_recluster_umap.h5ad


In [ ]:
work = sc.read_h5ad(os.path.join(outdir, "mono_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 33314 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [ ]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
   "Classical monocyte": ['CD14', 'FCN1', 'LST1', 'VCAN', 'CCR2', 'SELL', 'CTSS'],
    "Intermediate monocyte": [ 'MS4A7', 'HLA-DPA1', 'HLA-DPB1', 'ITGAX', 'IRF7', 'STAT1'],# CD14,FCGR3A,
    "Non-classical monocyte": ['FCGR3A', 'CX3CR1', 'NR4A1', 'ITGAL', 'PDE4B', 'LILRB1', 'HSPA1A']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mono_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mono_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Classical monocyte ['CD14', 'FCN1', 'LST1', 'VCAN', 'CCR2', 'SELL', 'CTSS']
Intermediate monocyte ['MS4A7', 'HLA-DPA1', 'HLA-DPB1', 'ITGAX', 'IRF7', 'STAT1']
Non-classical monocyte ['FCGR3A', 'CX3CR1', 'NR4A1', 'ITGAL', 'PDE4B', 'LILRB1', 'HSPA1A']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [ ]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mono_cluster_level2_marker_summary.csv"))


                  major_marker  major_marker_frac  n_cells    cluster_label_clean
cluster                                                                          
0           Classical monocyte           0.994937     1185     Classical monocyte
1           Classical monocyte           0.940778     1182     Classical monocyte
2           Classical monocyte           0.655261     1169     Classical monocyte
3        Intermediate monocyte           0.615652     1150  Intermediate monocyte
4           Classical monocyte           0.560805     1143     Classical monocyte
5           Classical monocyte           0.722621     1114     Classical monocyte
6           Classical monocyte           0.974843     1113     Classical monocyte
7           Classical monocyte           0.983755     1108     Classical monocyte
8           Classical monocyte           0.786916     1070     Classical monocyte
9           Classical monocyte           0.995287     1061     Classical monocyte
10          Clas

In [ ]:
corrected_annotation = {
    
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
# manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
# manual_mask = manual_map.notna()
# work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Classical monocyte       29038
Intermediate monocyte     4276
Name: count, dtype: int64


In [ ]:
work.write_h5ad(os.path.join(outdir, "mono_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mono_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_mono_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_mono_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mono_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_mono_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

In [ ]:
####还有mast cell,先不做

# 合并

In [2]:
outdir="../../7.5_correct/0511_rename_noIAISR/output_allhuman/work_0513_2/"
adata = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_corrected_all.h5ad"))
adata

AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [3]:
adata.obs["cell_type_level1_corrected"].value_counts()

cell_type_level1_corrected
T cell                   355512
Macrophage               104714
B cell                    89606
Fibroblast                87730
Neutrophil                83219
Smooth muscle cell        72481
Endothelial cell          65117
Natural killer cell       53467
Dendritic cell            34542
Monocyte                  33314
Pericyte                  18985
Mast cell                  6959
Erythrocyte/Erythroid      6884
Basophil                   3169
Name: count, dtype: int64

In [ ]:
outdir="/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0514_rename_level2/output_allhuman/work_0514"

adata_bc = sc.read_h5ad(os.path.join(outdir, "bc/bc_scPoli_recluster_umap_level2.h5ad"))
adata_dc = sc.read_h5ad(os.path.join(outdir, "dc/dc_scPoli_recluster_umap_level2.h5ad"))
adata_mac = sc.read_h5ad(os.path.join(outdir, "mac/mac_scPoli_recluster_umap_level2.h5ad"))
adata_smc = sc.read_h5ad(os.path.join(outdir, "smc/smc_scPoli_recluster_umap_level2.h5ad"))
adata_tc = sc.read_h5ad(os.path.join(outdir, "tc/tc_scPoli_recluster_umap_level2.h5ad"))
adata_mono = sc.read_h5ad(os.path.join(outdir, "mono/mono_scPoli_recluster_umap_level2.h5ad"))
adata_ec = sc.read_h5ad(os.path.join(outdir, "ec/ec_scPoli_recluster_umap_level2.h5ad"))

In [ ]:
adata_list = [adata_bc, adata_dc,adata_mac,adata_smc,adata_tc,adata_ec,adata_mono]

In [ ]:
adata_concat = anndata.concat(adata_list, join='outer', fill_value=0.0)

In [ ]:
adata_concat.write(os.path.join(outdir,"scPoli_concat_level2.h5ad"))
adata_concat

AnnData object with n_obs × n_vars = 755286 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3', 'marker_score__B_cell', 'marker_score__Plasma_cell', 'marker_top', 'marker_top_score', 'marker_second_score', 'marker_margin', 'marker_disagree', 'cell_type_level2_marker', 'marker_score__cDC1', 'marker_score__cDC2', 'marker_score__Plasmacytoid_dendritic_cell', 'marker_score__Inflammatory_macrophage', 'marker_score__Foamy_macrophage', 'marker_score__Homeostatic_Resident_m

In [5]:
outdir="/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0514_rename_level2/output_allhuman/work_0514"
adata_concat = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_level2.h5ad"))
adata_concat

AnnData object with n_obs × n_vars = 755286 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3', 'marker_score__B_cell', 'marker_score__Plasma_cell', 'marker_top', 'marker_top_score', 'marker_second_score', 'marker_margin', 'marker_disagree', 'cell_type_level2_marker', 'marker_score__cDC1', 'marker_score__cDC2', 'marker_score__Plasmacytoid_dendritic_cell', 'marker_score__Inflammatory_macrophage', 'marker_score__Foamy_macrophage', 'marker_score__Homeostatic_Resident_m

In [6]:
adata_concat.obs_names

Index(['1_JD__AAACCCAAGGAGTCTG-1', '1_JD__AAACCCATCCTTCGAC-1',
       '1_JD__AAACGAAAGCGTCTCG-1', '1_JD__AAACGAACACTTCCTG-1',
       '1_JD__AAACGCTAGCATCCTA-1', '1_JD__AAACGCTCAAACTAAG-1',
       '1_JD__AAAGAACGTGACACAG-1', '1_JD__AAAGAACTCCGCGAGT-1',
       '1_JD__AAAGAACTCTCAATCT-1', '1_JD__AAAGGGCTCTCGAGTA-1',
       ...
       'GSE196943_11__TAAGCACGTGTGACCC_K_blood',
       'GSE196943_11__TCAGGGCGTTCTCGCT_K_blood',
       'GSE196943_11__TCGACGGGTAAGATTG_K_blood',
       'GSE196943_11__TCGATTTGTCAAGCGA_K_blood',
       'GSE196943_11__TGCCGAGTCGTAGGAG_K_blood',
       'GSE196943_11__TTCACCGTCCATTTCA_K_plaque',
       'GSE196943_11__CCCATTGGTATTTCCT_K_plaque',
       'GSE196943_10__GACCCTTGTGTTACAC_J_blood',
       'GSE196943_10__GTTCCGTTCGTTGCCT_J_blood',
       'GSE196943_7__GTTGTGACAGCGTTTA_G_plaque'],
      dtype='object', name='match_id', length=755286)

In [7]:
# Extract barcodes and cell_type_level2_marker values
barcodes = adata_concat.obs_names
cell_types_level2 = adata_concat.obs["cell_type_level2_marker"]

# Create the mapping
mapping = dict(zip(barcodes, cell_types_level2))

In [8]:
adata.obs_names

Index(['1_JD__AAACCCAAGAGGTTAT-1', '1_JD__AAACCCAAGCAACAAT-1',
       '1_JD__AAACCCAAGGAGTCTG-1', '1_JD__AAACCCACAACTCATG-1',
       '1_JD__AAACCCACAGCTCTGG-1', '1_JD__AAACCCAGTCAAGGCA-1',
       '1_JD__AAACCCAGTCAATCTG-1', '1_JD__AAACCCAGTCACCCTT-1',
       '1_JD__AAACCCAGTGCATTAC-1', '1_JD__AAACCCAGTGCGGCTT-1',
       ...
       'GSE196943_8__ATAACGCAGTACGCCC_H_plaque',
       'GSE196943_8__CACATAGGTGTCTGAT_H_plaque',
       'GSE196943_8__CCATTCGAGTAGCCGA_H_plaque',
       'GSE196943_8__GAATAAGAGAAAGTGG_H_plaque',
       'GSE196943_8__GACCAATCATGGTAGG_H_plaque',
       'GSE196943_8__GACCTGGGTCTTCAAG_H_plaque',
       'GSE196943_8__TGGGCGTTCTTCTGGC_H_plaque',
       'GSE196943_8__GAGCAGATCAGAGGTG_H_plaque',
       'GSE196943_8__GAGTCCGGTTCTGGTA_H_plaque',
       'GSE196943_8__TCATTTGAGTTGTCGT_H_plaque'],
      dtype='object', name='match_id', length=1015699)

In [9]:
# init new column
adata.obs["cell_type_level2"] = "no map"

In [10]:
### more fast
if "cell_type_level2" not in adata.obs.columns:
    adata.obs["cell_type_level2"] = pd.NA

mapped = adata.obs.index.to_series().map(mapping)
mask = mapped.notna()
adata.obs.loc[mask, "cell_type_level2"] = mapped[mask].to_numpy()

In [11]:
adata.obs["cell_type_level2"].value_counts()

cell_type_level2
no map                                   260413
CD4 T cell                               228073
CD8 T cell                               127439
B cell                                    78088
Homeostatic/Resident macrophage           69184
Smooth muscle cell                        57935
Lymphatic endothelial cell                43449
Classical monocyte                        29038
Arterial endothelial cell                 21668
cDC1                                      20734
Inflammatory macrophage                   20256
Fibromyocyte                              14546
Foamy macrophage                          12463
Plasma cell                               11518
cDC2                                       9944
Intermediate monocyte                      4276
Plasmacytoid dendritic cell                3864
Other macrophage                           1696
Matrix-remodeling/SMC-like macrophage      1115
Name: count, dtype: int64

In [ ]:
# adata = adata[adata.obs['cell_type_level2_marker'] != 'Undefine'].copy()
# adata

AnnData object with n_obs × n_vars = 1017964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [12]:
mask = adata.obs["cell_type_level2"] == "no map"
adata.obs.loc[mask, "cell_type_level2"] = adata.obs.loc[mask, "cell_type_level1_corrected"]

In [13]:
adata.obs["cell_type_level2"].value_counts()

cell_type_level2
CD4 T cell                               228073
CD8 T cell                               127439
Fibroblast                                87730
Neutrophil                                83219
B cell                                    78088
Homeostatic/Resident macrophage           69184
Smooth muscle cell                        57935
Natural killer cell                       53467
Lymphatic endothelial cell                43449
Classical monocyte                        29038
Arterial endothelial cell                 21668
cDC1                                      20734
Inflammatory macrophage                   20256
Pericyte                                  18985
Fibromyocyte                              14546
Foamy macrophage                          12463
Plasma cell                               11518
cDC2                                       9944
Mast cell                                  6959
Erythrocyte/Erythroid                      6884
Intermediate monocyte  

In [14]:
adata.write(os.path.join(outdir,"scPoli_concat_level2_marker_allhuman.h5ad"))
adata

AnnData object with n_obs × n_vars = 1015699 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names', 'ensembl_id'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'